# **1. IMPORT LIBRARY**

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# **2. LOAD DATASET**

In [2]:
DATA_PATH = Path("../data/processed")
users = pd.read_csv(DATA_PATH / "users_clean.csv")

kolom_dipakai = ["user_id", "text", "created_at", "label", "source_folder"]
tweets = pd.read_csv(DATA_PATH / "tweets_clean.csv", usecols=kolom_dipakai)

users = users.rename(columns={"id": "user_id"})

In [3]:
users["created_at"] = pd.to_datetime(
    users["created_at"],
    errors="coerce"
)

tweets["created_at"] = pd.to_datetime(
    tweets["created_at"],
    errors="coerce"
)

# **3. FEATURE A**

In [4]:
reference_date = tweets["created_at"].max()
users["created_at"] = pd.to_datetime(users["created_at"], errors="coerce")
tweets["created_at"] = pd.to_datetime(tweets["created_at"], errors="coerce")

if users["created_at"].dt.tz is not None:
    users["created_at"] = users["created_at"].dt.tz_localize(None)

if tweets["created_at"].dt.tz is not None:
    tweets["created_at"] = tweets["created_at"].dt.tz_localize(None)

reference_date = tweets["created_at"].max()
users["account_age_days"] = (reference_date - users["created_at"]).dt.days
users["account_age_days"] = (reference_date - users["created_at"]).dt.days
users["follow_ratio"] = users["followers_count"] / (users["friends_count"] + 1)
users["tweets_per_day"] = users["statuses_count"] / (users["account_age_days"] + 1)
users["favorites_per_day"] = users["favourites_count"] / (users["account_age_days"] + 1)
users["listed_ratio"] = users["listed_count"] / (users["followers_count"] + 1)

feature_A = users[[
    "user_id", "source_folder", "followers_count", "friends_count", "statuses_count",
    "listed_count", "favourites_count", "verified", "protected",
    "geo_enabled", "account_age_days", "follow_ratio", "tweets_per_day",
    "favorites_per_day", "listed_ratio", "label"
]]

tweets["tweet_length"] = tweets["text"].str.len()
tweets["hashtag_count"] = tweets["text"].str.count("#")
tweets["url_count"] = tweets["text"].str.count("http")
tweets["mention_count"] = tweets["text"].str.count("@")
tweets["is_retweet"] = tweets["text"].str.startswith("RT")

tweets = tweets.sort_values(["user_id", "created_at"])
tweets["interval"] = tweets.groupby("user_id")["created_at"].diff().dt.total_seconds()

individual_activity = (
    tweets.groupby("user_id")
    .agg(
        tweet_count=("text", "size"),
        avg_tweet_length=("tweet_length", "mean"),
        avg_hashtag_count=("hashtag_count", "mean"),
        avg_url_count=("url_count", "mean"),
        avg_mention_count=("mention_count", "mean"),
        avg_posting_interval=("interval", "mean"),
        retweet_ratio=("is_retweet", "mean"),
    )
    .reset_index()
)

feature_A = feature_A.merge(individual_activity, on="user_id", how="left")

# **4. FEATURE B**

In [5]:
# 5a. Synchronized Hashtag Usage
# Ekstrak hashtag mentah dari teks
def extract_hashtags(text):
    if pd.isna(text):
        return []
    return re.findall(r"#\w+", text.lower())

tweets["hashtags_list"] = tweets["text"].apply(extract_hashtags)
tweets["hour_bucket"] = tweets["created_at"].dt.floor("h")

exploded = tweets[["user_id", "hour_bucket", "hashtags_list"]].explode("hashtags_list")
exploded = exploded.dropna(subset=["hashtags_list"])

hashtag_activity = (
    exploded.groupby(["hashtags_list", "hour_bucket"])["user_id"]
    .nunique()
    .reset_index(name="users_same_hashtag_hour")
)

exploded = exploded.merge(hashtag_activity, on=["hashtags_list", "hour_bucket"], how="left")
sync_hashtag_score = (
    exploded.groupby("user_id")["users_same_hashtag_hour"]
    .mean()
    .reset_index(name="sync_hashtag_score")
)

# 5b. Duplicate Content Score
tweets["text_normalized"] = tweets["text"].str.strip().str.lower()

text_user_count = (
    tweets.groupby("text_normalized")["user_id"]
    .nunique()
    .reset_index(name="users_with_same_text")
)
tweets = tweets.merge(text_user_count, on="text_normalized", how="left")

duplicate_score = (
    tweets.groupby("user_id")["users_with_same_text"]
    .mean()
    .reset_index(name="duplicate_content_score")
)

# 5c. Gabung Feature B
feature_B = sync_hashtag_score.merge(duplicate_score, on="user_id", how="outer")
feature_B = feature_B.fillna(0)

POSTING INTERVAL

In [6]:
# #SORT TWEET
# tweets = tweets.sort_values([
#     "user_id",
#     "created_at"
# ])

# #TIME INTERVAL
# tweets["interval"] = (

#     tweets
#     .groupby("user_id")["created_at"]
#     .diff()

#     .dt.total_seconds()
# )

# #AVG
# posting_interval = (

#     tweets

#     .groupby("user_id")["interval"]

#     .mean()

#     .reset_index()
# )

RETWEET RATIO

In [7]:
# tweets["is_retweet"] = (
#     tweets["text"]
#     .str.startswith("RT")
# )

# retweet_ratio = (

#     tweets

#     .groupby("user_id")["is_retweet"]

#     .mean()

#     .reset_index()
# )

MERGE FEATURE B

In [8]:
# feature_B = (
#     tweet_count
#     .merge(tweet_length, on="user_id")
#     .merge(hashtag_feature, on="user_id")
#     .merge(url_feature, on="user_id")
#     .merge(mention_feature, on="user_id")
#     .merge(posting_interval, on="user_id")
#     .merge(retweet_ratio, on="user_id")
# )

# **5. MERGE A+B**

In [9]:
feature_AB = feature_A.merge(feature_B, on="user_id", how="left")
feature_AB = feature_AB.fillna(0)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10692\1669825373.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  feature_AB = feature_AB.fillna(0)


# **6. SAVE**

In [10]:
OUTPUT = Path("../data/processed")   
OUTPUT.mkdir(parents=True, exist_ok=True)

feature_A.to_csv(OUTPUT / "feature_A.csv", index=False)
feature_B.to_csv(OUTPUT / "feature_B.csv", index=False)
feature_AB.to_csv(OUTPUT / "feature_AB.csv", index=False)

print("Feature A shape:", feature_A.shape)
print("Feature B shape:", feature_B.shape)
print("Feature AB shape:", feature_AB.shape)

Feature A shape: (14368, 23)
Feature B shape: (10197, 3)
Feature AB shape: (14368, 25)
